In [ ]:
from darts.dataprocessing.transformers import Scaler, StaticCovariatesTransformer
from darts.models import TFTModel
import pandas as pd
import matplotlib.pyplot as plt

# ==================================================
# 1. Take just 1 series
# ==================================================
single_target = [all_targets[0]]
single_future_covs = [all_future_covs[0]] if all_future_covs else None

val_cutoff  = pd.Timestamp('2025-04-01')
test_cutoff = pd.Timestamp('2025-10-01')

# ==================================================
# 2. Split
# ==================================================
train_single = [s.split_before(val_cutoff)[0] for s in single_target]
val_single   = [s.split_before(test_cutoff)[0] for s in single_target]  

if single_future_covs:
    train_future_single = [s.split_before(val_cutoff)[0] for s in single_future_covs]
    val_future_single   = [s.split_before(test_cutoff)[0] for s in single_future_covs]
else:
    train_future_single = None
    val_future_single   = None

# ==================================================
# 3. Scale
# ==================================================
target_scaler = Scaler()
train_single_scaled = target_scaler.fit_transform(train_single)
val_single_scaled   = target_scaler.transform(val_single)

static_scaler = StaticCovariatesTransformer()
train_single_scaled = static_scaler.fit_transform(train_single_scaled)
val_single_scaled   = static_scaler.transform(val_single_scaled)

if single_future_covs:
    future_cov_scaler = Scaler()
    train_future_single_scaled = future_cov_scaler.fit_transform(train_future_single)
    val_future_single_scaled   = future_cov_scaler.transform(val_future_single)
else:
    train_future_single_scaled = None
    val_future_single_scaled   = None

# ==================================================
# 4. Loss tracking callback
# ==================================================
import pytorch_lightning as pl

class LossTracker(pl.Callback):
    def __init__(self):
        self.train_losses = []
        self.val_losses   = []

    def on_train_epoch_end(self, trainer, pl_module):
        loss = trainer.callback_metrics.get("train_loss")
        if loss:
            self.train_losses.append(loss.item())
            print(f"Epoch {trainer.current_epoch+1} | train_loss: {loss.item():.6f}")

    def on_validation_epoch_end(self, trainer, pl_module):
        loss = trainer.callback_metrics.get("val_loss")
        if loss:
            self.val_losses.append(loss.item())

loss_tracker = LossTracker()

# ==================================================
# 5. Overfit model config
# — no dropout
# — no early stopping
# — small batch size to get many gradient updates
# — high epochs
# ==================================================
categorical_embedding_sizes = {
    col: len(encoders[col].classes_) for col in static_cols
}

overfit_model = TFTModel(
    input_chunk_length=12,
    output_chunk_length=6,
    hidden_size=64,       # slightly bigger than original to ensure capacity
    lstm_layers=1,
    num_attention_heads=2,
    dropout=0.0,          # ← no dropout for overfit test
    batch_size=8,         # ← very small batch = more gradient updates per epoch
    n_epochs=100,         # ← many epochs to force overfit
    categorical_embedding_sizes=categorical_embedding_sizes,
    add_encoders=add_encoders,
    pl_trainer_kwargs={
        "accelerator": "gpu",
        "devices": [0],
        "precision": "32",         # ← full precision, no mixed precision instability
        "enable_progress_bar": True,
        "callbacks": [loss_tracker],  # ← no early stopping
    },
    random_state=42,
)

# ==================================================
# 6. Fit on single series
# ==================================================
overfit_model.fit(
    series=train_single_scaled,
    val_series=val_single_scaled,
    future_covariates=train_future_single_scaled,
    val_future_covariates=val_future_single_scaled,
    verbose=True,
)

# ==================================================
# 7. Plot loss curve
# ==================================================
plt.figure(figsize=(12, 5))
plt.plot(loss_tracker.train_losses, label="Train Loss", color="blue")
if loss_tracker.val_losses:
    plt.plot(loss_tracker.val_losses, label="Val Loss", color="orange")
plt.xlabel("Epoch")
plt.ylabel("Quantile Loss")
plt.title("Overfit Test — Single Series Loss Curve")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

# ==================================================
# 8. Print final result
# ==================================================
print(f"\nInitial train_loss: {loss_tracker.train_losses[0]:.6f}")
print(f"Final  train_loss: {loss_tracker.train_losses[-1]:.6f}")
print(f"Reduction: {((loss_tracker.train_losses[0] - loss_tracker.train_losses[-1]) / loss_tracker.train_losses[0]) * 100:.1f}%")